# 08 Final strategy: simulated live tracking (2026-10-01 to 2026-10-20)

A **new** account, separate from the historical backtest in 07: USD 100,000 on 2026-10-01, split once into USD 90,000 (US, V2) and the SGD equivalent of USD 10,000 (Singapore, ES3 + A35). Signals may use earlier data, but no holdings, cash or NAV are carried over from 07.

Same rules as 07 (see `portfolio_accounting.py`): each account rebalances at its own next valid opening with an earlier signal, missing quotes block trading, accounts are reset to 90/10 only if the Singapore share leaves 10% ± 1pt (none expected in this window). This is a simulation on daily prices, not broker execution.

**Outputs:** `results/tables/final_live_summary.csv`, `final_live_trades.csv`, `final_live_daily_positions.csv`, `final_live_current_holdings.csv`, `results/signals/final_live_next_open_target.csv` and `results/figures/final_live_equity.png`. In the target file, `target_within_sleeve` is the model weight inside each account and `indicative_total_weight` = current account share × that weight.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
ROOT=Path.cwd().resolve()
while not (ROOT/'pipeline/00_engine.ipynb').is_file() and ROOT!=ROOT.parent:
    ROOT=ROOT.parent
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
import portfolio_accounting as pa

TABLE_DIR, SIGNAL_DIR, FIG_DIR=[ROOT/'results'/s for s in ('tables','signals','figures')]


def main(live_start=pa.LIVE_START,live_end=pa.LIVE_END,capital=pa.LIVE_CAPITAL,
         strategy='v2',rebalance='daily',save=True):
    us,sg=pa.load_markets()
    w_us,w_sg=pa.strategy_weights(strategy)
    result=pa.run_portfolio(us,sg,w_us,w_sg,live_start,live_end,capital,rebalance=rebalance,rebalance_band=0.01)
    daily,trades=result['daily'],result['trades']
    end=daily.index[-1]
    if w_us.index.max()<min(end,us.closes.index.max()):
        raise ValueError('US signals are stale. Re-run 02, 05, and 06.')
    if w_sg.index.max()<min(end,sg.closes.index.max()):
        raise ValueError('Singapore signals are stale. Re-run 03.')
    if end<pd.Timestamp(live_end):print(f'Window incomplete: planned through {live_end}; ledger through {end.date()}.')
    summary=[]
    eq=daily.equity
    summary.append(dict(Strategy='Final V2 + Singapore (90/10 start; ±1pt threshold)',
        Period=f'{live_start}~{end.date()}',Valuation_days=len(daily)-1,
        Ending_equity=round(eq.iloc[-1],2),Total_return=f'{eq.iloc[-1]/capital-1:+.2%}',
        Max_drawdown=f'{(eq/eq.cummax()-1).min():.2%}',Trade_count=len(trades),
        Total_cost_USD=round(trades.cost_usd.sum(),2),US_sleeve_share=daily.US_sleeve_pct.iloc[-1],
        SG_sleeve_share=daily.SG_sleeve_pct.iloc[-1],SG_quote_date=daily.SG_quote_date.iloc[-1],
        Valuation_status='Carried SG quote; source date shown' if daily.SG_quote_date.iloc[-1]<end else 'Source closing quotes available'))
    benchmark_equities={}
    for t in ['SPY','QQQ','TQQQ']:
        weights=pd.DataFrame({t:1.0},index=us.closes.index)
        b=pa.run_sleeve(us,weights,daily.index,capital,rebalance='buy_hold')
        e=b['daily'].equity;benchmark_equities[t]=e
        summary.append(dict(Strategy='Buy and hold '+t+' (entry fee included)',
            Period=f'{live_start}~{end.date()}',Valuation_days=len(e)-1,
            Ending_equity=round(e.iloc[-1],2),Total_return=f'{e.iloc[-1]/capital-1:+.2%}',
            Max_drawdown=f'{(e/e.cummax()-1).min():.2%}',Trade_count=len(b['trades']),
            Total_cost_USD=round(b['trades'].cost_usd.sum(),2)))
    summary=pd.DataFrame(summary)
    current=pa.holdings(result);latest=pa.latest_targets(result)
    display(summary);display(current);display(latest)
    print('Each sleeve trades only at its own valid source opening. Filled marks never execute.')
    print('Latest statuses:',daily[['US_status','SG_status','US_quote_date','SG_quote_date']].tail(1).to_dict('records'))
    print('Transfers in window:',len(result['transfers']),'(threshold +/-1pt). SGD cash remains in the Singapore account.')
    fig,ax=plt.subplots(figsize=(10,4))
    (eq/capital).plot(ax=ax,label='Final independent sleeves')
    for t,e in benchmark_equities.items():(e/capital).plot(ax=ax,label=t)
    ax.legend();ax.set_title('Simulated live tracking; 90/10 start, +/-1pt threshold');fig.tight_layout()
    prefix='final_live' if strategy=='v2' else 'live_oct'
    if save:
        for p in [TABLE_DIR,SIGNAL_DIR,FIG_DIR]:p.mkdir(parents=True,exist_ok=True)
        trades.to_csv(TABLE_DIR/f'{prefix}_trades.csv',index=False,encoding='utf-8-sig')
        daily.to_csv(TABLE_DIR/f'{prefix}_daily_positions.csv',encoding='utf-8-sig')
        current.to_csv(TABLE_DIR/f'{prefix}_current_holdings.csv',encoding='utf-8-sig')
        summary.to_csv(TABLE_DIR/f'{prefix}_summary.csv',index=False,encoding='utf-8-sig')
        latest.to_csv(SIGNAL_DIR/f'{prefix}_next_open_target.csv',encoding='utf-8-sig')
        fig.savefig(FIG_DIR/f'{prefix}_equity.png',dpi=150,bbox_inches='tight')
    plt.show()
    return {'portfolio':result,'daily':daily,'trades':trades,'summary':summary,'latest':latest,'holdings':current}


In [ ]:
LIVE_START = pa.LIVE_START
LIVE_END = pa.LIVE_END
CAPITAL = pa.LIVE_CAPITAL
results_live = main(LIVE_START, LIVE_END, CAPITAL, save=True)
